In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
import pickle
import os

In [2]:
print(os.getcwd())

G:\MAIA\modules\yield_prediction


In [10]:
df = pd.read_csv("crop_yield_lstm.csv")
print(f"Dataset shape: {df.shape}")

Dataset shape: (1000, 7)


In [11]:
df.head()

,Temperature,Rainfall,Humidity,Soil Type,Weather Condition,Crop Type,Yield (tons/hectare)
0,22.490802,185.132929,53.085284,Sandy,Sunny,Barley,2.818937
1,34.014286,541.900947,52.348940,Loamy,Sunny,Corn,8.014166
2,29.639879,872.945836,85.312729,Peaty,Rainy,Wheat,9.249868
3,26.973170,732.224886,52.477310,Sandy,Sunny,Soybeans,7.947481
4,18.120373,806.561148,53.597486,Clay,Stormy,Barley,6.262616


In [12]:
FEATURES = ['Rainfall', 'Temperature', 'Humidity']
TARGET   = 'Yield (tons/hectare)'

In [13]:
df[FEATURES]

,Rainfall,Temperature,Humidity
0,185.132929,22.490802,53.085284
1,541.900947,34.014286,52.348940
2,872.945836,29.639879,85.312729
3,732.224886,26.973170,52.477310
4,806.561148,18.120373,53.597486
...,...,...,...
995,656.955156,16.831641,83.264788
996,956.614621,33.346272,47.863660
997,68.958016,17.736373,55.489393
998,57.054721,34.004747,54.502277


In [14]:
scaler = MinMaxScaler()
df[FEATURES] = scaler.fit_transform(df[FEATURES])
# pickle.dump(scaler, open(YIELD_SCALER_PATH, "wb"))
# print(f"✅ Scaler saved: {YIELD_SCALER_PATH}")

In [18]:
df[FEATURES].values

array([[0.18260941, 0.37173493, 0.26226862],
       [0.54073995, 0.95075462, 0.2475094 ],
       [0.87304912, 0.73095408, 0.90823268],
       ...,
       [0.06599082, 0.13283943, 0.31045637],
       [0.05404206, 0.95027532, 0.29067069],
       [0.28003421, 0.44355353, 0.87331564]])

In [19]:
SEQ_LEN = 4

def make_sequences(data, target, seq=SEQ_LEN):
    X, y = [], []
    for i in range(len(data) - seq):
        X.append(data[i:i+seq])
        y.append(target[i+seq])
    return np.array(X), np.array(y)

In [20]:
X, y = make_sequences(
    df[FEATURES].values,
    df[TARGET].values
)

In [21]:
split  = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f"Train: {X_train.shape} | Test: {X_test.shape}")

Train: (796, 4, 3) | Test: (200, 4, 3)


In [22]:
# ── Build LSTM Model ─────────────────────────────
model = tf.keras.Sequential([
    tf.keras.layers.LSTM(64,
        input_shape=(SEQ_LEN, len(FEATURES)),
        return_sequences=True),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1)
])

In [23]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

In [24]:
model.summary()

_________________________________________________________________


In [25]:
from config import YIELD_LSTM_PATH

# ── Train ────────────────────────────────────────
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "yield_lstm.keras",
    save_best_only=True,
    monitor='val_loss',
    verbose=1
)

In [26]:
model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    batch_size=16,
    callbacks=[checkpoint],
    verbose=1
)

In [31]:
r=model.evaluate(X_test, y_test)

7/7 [==============================] - 1s 21ms/step - loss: 5.7762 - mae: 2.1107


In [27]:
X_train

array([[[0.18260941, 0.37173493, 0.26226862],
        [0.54073995, 0.95075462, 0.2475094 ],
        [0.87304912, 0.73095408, 0.90823268],
        [0.73179075, 0.59696013, 0.25008244]],

       [[0.54073995, 0.95075462, 0.2475094 ],
        [0.87304912, 0.73095408, 0.90823268],
        [0.73179075, 0.59696013, 0.25008244],
        [0.80641091, 0.15213426, 0.27253516]],

       [[0.87304912, 0.73095408, 0.90823268],
        [0.73179075, 0.59696013, 0.25008244],
        [0.80641091, 0.15213426, 0.27253516],
        [0.65806875, 0.15211002, 0.76105393]],

       ...,

       [[0.67959506, 0.60981908, 0.26009997],
        [0.0386015 , 0.01354472, 0.43794369],
        [0.08188529, 0.8717761 , 0.59485249],
        [0.71582852, 0.93206676, 0.07323036]],

       [[0.0386015 , 0.01354472, 0.43794369],
        [0.08188529, 0.8717761 , 0.59485249],
        [0.71582852, 0.93206676, 0.07323036],
        [0.06912908, 0.56326926, 0.62369801]],

       [[0.08188529, 0.8717761 , 0.59485249],
        [0.

In [28]:
last_4_weeks=[[650,32,70],[600,33,68],[700,31,75],[620,32,72]]

In [29]:
seq_scaled = scaler.transform(last_4_weeks)
seq_tensor = np.array([seq_scaled])
model3     = float(model.predict(seq_tensor, verbose=0)[0][0])

G:\MAIA\.venv\lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [30]:
model3

6.395395278930664